In [0]:
current_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
display(current_notebook_path)

In [0]:
#returncode = dbutils.notebook.run("/Users/pavandecourse2@gmail.com/DataEngineering-Project-1/MiniProject-3/Homework-2", 300)

In [0]:
productsFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/products.csv"
display(productsFilePath)

ordersFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/orders.csv"
display(ordersFilePath)

customersFilePath = "abfss://miniproject@unluckystorageaccount2.dfs.core.windows.net/customers.csv"
display(customersFilePath)

In [0]:
productsDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(productsFilePath)
#display(productsDF)
ordersDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(ordersFilePath)
#display(ordersDF)
customersDF = spark.read.format("csv").option("header",True).option("inferSchema",True).load(customersFilePath)
#display(customersDF)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
#2.	Group orders by STATUS and calculate: order count, total quantity, and average quantity.
total_qty_sold = ordersDF.agg(sum("QTY").alias("Total_Quantity_Sold"))
display(total_qty_sold)

In [0]:
ordersDF.printSchema()

In [0]:
#3.	Group products by CATEGORY and find the average PRICE for each category.
orders_count = ordersDF.orderBy("STATUS").agg(
    sum("QTY").alias("SUM_of_QTY"),
    count("ORDERID").alias("order_count"),
    sum("QTY").alias("total_QTY"),
    avg("QTY").alias("average quantity")

)
display(orders_count)

In [0]:
#4.	Join orders with products on PRODID (inner join) and display ORDERID, PNAME, CATEGORY, QTY, and PRICE.
ordersDF_Join=ordersDF.join(productsDF,ordersDF.PRODID == productsDF.PRODID ,"inner").select("ORDERID","PNAME","CATEGORY","QTY","PRICE")

display(ordersDF_Join)

In [0]:
#5.	Using a left join, find products from the products table that have NEVER been ordered (Hint: join products with orders on PRODID, then filter where the order columns are null).
ordersDF_LeftJoin=ordersDF.join(productsDF,ordersDF.PRODID == productsDF.PRODID ,"left").filter(ordersDF.ORDERID.isNull())

display(ordersDF_LeftJoin)


In [0]:
#6.	Join orders with customers on CUSTID, then using Window with partitionBy('CUSTID') and orderBy(col('ORDERDATE')), add an ORDER_SEQ column (row_number) showing each customer's order sequence.

windowsspec=Window.partitionBy(ordersDF["CUSTID"]).orderBy(col("ORDERDATE"))

ordersDF_Join = ordersDF.join(customersDF,ordersDF.CUSTID == customersDF.CUSTID,"inner").withColumn("ORDER_SEQ",row_number().over(windowsspec).alias("row_number"))

display(ordersDF_Join)

In [0]:
#6.	Join orders with customers on CUSTID, then using Window with partitionBy('CUSTID') and orderBy(col('ORDERDATE')), add an ORDER_SEQ column (row_number) showing each customer's order sequence.

windowsspec = Window.partitionBy(ordersDF["CUSTID"]).orderBy(col("ORDERDATE"))

ordersDF_Join = ordersDF.join(customersDF, ordersDF.CUSTID == customersDF.CUSTID, "inner") \
    .withColumn("ORDER_SEQ", row_number().over(windowsspec))

display(ordersDF_Join
)    

In [0]:
#7.	Using Window with partitionBy('CATEGORY') (after joining orders with products) and orderBy(col('QTY').desc()), add a RANK and DENSE_RANK column for quantity ordered within each category.

windowsspec= Window.partitionBy("CATEGORY").orderBy(col("QTY").desc())
ordersDF_Join = ordersDF.join(productsDF,ordersDF.PRODID == productsDF.PRODID,"inner").withColumn("RANK",rank().over(windowsspec)).withColumn("DENSE_RANK",dense_rank().over(windowsspec))
display(ordersDF_Join)

In [0]:
productsDF.printSchema()

In [0]:
#8.	Find the highest quantity order placed for each product category (Hint: filter where RANK = 1 from the previous question).

windowsspec = Window.partitionBy("CATEGORY").orderBy(col("QTY").desc())
ordersDF_Join = ordersDF.join(productsDF,ordersDF.PRODID == productsDF.PRODID,"inner").withColumn("RANK",row_number().over(windowsspec))
 
display(ordersDF_Join.filter(col("RANK") ==1))

 

In [0]:
#9.Create a pivot table showing the count of orders per STATUS (rows) across each CATEGORY (columns 'Electronics', 'Furniture', 'Stationery'), filling nulls with 0. (Hint: join orders with products first.)


windowsspec = Window.partitionBy("CATEGORY").orderBy(col("QTY").desc())
ordersDF_Join = ordersDF.join(productsDF,ordersDF.PRODID == productsDF.PRODID,"inner")#.withColumn("RANK" ,row_number().over(windowsspec))
 
pivot_df =ordersDF_Join.groupBy("STATUS").pivot("CATEGORY").agg(count("CATEGORY")).fillna(0)

display(pivot_df)

In [0]:
#9.	Create a pivot table showing the count of orders per STATUS (rows) across each CATEGORY (columns 'Electronics', 'Furniture', 'Stationery'), filling nulls with 0. (Hint: join orders with products first.)

orders_products_df = ordersDF.join(productsDF, ordersDF.PRODID == productsDF.PRODID, "inner")

pivot_df = orders_products_df.groupBy("STATUS") \
    .pivot("CATEGORY", ["Electronics", "Furniture", "Stationery"]) \
    .agg(count("ORDERID")) \
    .fillna(0)

display(pivot_df)